# Data integration

This notebook combines transaction, account, customer and merchant data into a single enriched analytical dataset

## Objectives

- Practice the different types of table joins
- Validate primary and foreign keys
- Join transactions with accounts
- Join accounts with customers
- Join transactions with merchants
- Detect unmatched records
- Prevent accidental row duplication
- Create cross table analytical features
- Compare merge, join and concat

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
current_directory = Path.cwd()

In [4]:
if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [5]:
transactions_path = (
    project_root
    / "data"
    / "processed"
    / "transactions_features.csv"
)

accounts_path = (
    project_root
    / "data"
    / "raw"
    / "accounts.csv"
)

customers_path = (
    project_root
    / "data"
    / "raw"
    / "customers.csv"
)

merchants_path = (
    project_root
    / "data"
    / "raw"
    / "merchants.csv"
)

In [7]:
paths_exist = pd.Series(
    {
        "transactions": transactions_path.exists(),
        "accounts": accounts_path.exists(),
        "customers": customers_path.exists(),
        "merchants": merchants_path.exists(),
    },
    name="exists",
)

paths_exist.all()

np.True_

## Loading datasets

The transformed transaction dataset is combined with the three reference tables: accounts, customers and merchants

In [10]:
transactions = pd.read_csv(transactions_path, parse_dates=["timestamp"])
accounts = pd.read_csv(accounts_path, parse_dates=["opening_date"])
customers = pd.read_csv(customers_path, parse_dates=["signup_date"])
merchants = pd.read_csv(merchants_path)

In [11]:
dataset_shapes = pd.DataFrame(
    {
        "rows": [
            len(transactions),
            len(accounts),
            len(customers),
            len(merchants),
        ],
        "columns": [
            len(transactions.columns),
            len(accounts.columns),
            len(customers.columns),
            len(merchants.columns),
        ],
    },
    index=[
        "transactions",
        "accounts",
        "customers",
        "merchants",
    ],
)

dataset_shapes

,rows,columns
transactions,52,24
accounts,16,6
customers,12,5
merchants,15,5


In [13]:
# Before joining tables primary keys are checked for uniqueness and foreign keys are reviewed

key_validation = pd.Series(
    {
        "transaction_id_unique":(transactions["transaction_id"].is_unique),
        "account_id_unique":(accounts["account_id"].is_unique),
        "customer_id_unique":(customers["customer_id"].is_unique),
        "merchant_id_unique":(merchants["merchant_id"].is_unique),
    },
    name = "is_unique"
)

key_validation.all()

np.True_

In [14]:
# Check for columns with repeated names

set(transactions.columns).intersection(accounts.columns)

{'account_id', 'currency', 'status'}

In [15]:
accounts_for_merge = accounts.rename(
    columns = {
        "currency" : "account_currency",
        "status" : "account_status"
    }
)

In [16]:
accounts_for_merge.columns

Index(['account_id', 'customer_id', 'account_type', 'account_currency',
       'opening_date', 'account_status'],
      dtype='str')

In [17]:
customers_for_merge = customers.rename(
    columns = {"country": "customer_country"}
)

In [18]:
merchants_for_merge = merchants.rename(
    columns={
        "category": "merchant_category",
        "country": "merchant_country",
    }
)

In [19]:
merchants_for_merge.columns

Index(['merchant_id', 'merchant_name', 'merchant_category', 'merchant_country',
       'online'],
      dtype='str')

# Transactions and accounts merge

Each transaction belongs to one account while one account may appear in many transactions.

- This is a many to one relationship

In [28]:
transactions_accounts = transactions.merge(
    accounts_for_merge,
    on = "account_id",
    how = "left",
    validate = "many_to_one",
    indicator = "account_merge"
)

In [29]:
transactions.shape

(52, 24)

In [30]:
transactions_accounts.shape

(52, 30)

In [31]:
transactions_accounts["account_merge"
].value_counts(dropna=False)

account_merge
both          52
left_only      0
right_only     0
Name: count, dtype: int64

In [34]:
unmatched_accounts = transactions_accounts.loc[transactions_accounts["account_merge"] != "both", 
["transaction_id", "account_id", "account_merge"]]

unmatched_accounts.empty

True

# Transactions and customers merge

The account table provides the customer_id which is used to add customer information to every transaction

In [35]:
transactions_customers = transactions_accounts.merge(
    customers_for_merge,
    on = "customer_id",
    how = "left",
    validate = "many_to_one",
    indicator = "customer_merge"
)

In [36]:
transactions_customers["customer_merge"].value_counts(dropna = False)

customer_merge
both          52
left_only      0
right_only     0
Name: count, dtype: int64

In [38]:
transactions_customers[
    [
        "transaction_id",
        "customer_id",
        "age",
        "customer_country",
        "signup_date",
        "customer_segment",
    ]
].head()


,transaction_id,customer_id,age,customer_country,signup_date,customer_segment
0,T0001,C001,29,Spain,2023-01-15,Standard
1,T0002,C002,42,Spain,2022-06-03,Premium
2,T0003,C003,23,France,2024-02-11,Student
3,T0004,C004,35,Germany,2021-09-27,Premium
4,T0005,C005,31,Spain,2023-07-19,Standard


# Transactions and merchants merge

Each transaction references one merchant; merchant information is added through merchant_id

In [39]:
transactions_enriched = transactions_customers.merge(
    merchants_for_merge,
    on = "merchant_id",
    how = "left",
    validate = "many_to_one",
    indicator = "merchant_merge"
)

### Transactions audit

In [41]:
merge_audit = pd.Series(
    {
        "original_transaction_rows": len(
            transactions
        ),
        "rows_after_account_merge": len(
            transactions_accounts
        ),
        "rows_after_customer_merge": len(
            transactions_customers
        ),
        "rows_after_merchant_merge": len(
            transactions_enriched
        ),
        "unmatched_accounts": len(
            unmatched_accounts
        ),
        "transaction_ids_remain_unique": (
            transactions_enriched[
                "transaction_id"
            ].is_unique
        ),
    },
    name="value",
)

merge_audit

original_transaction_rows          52
rows_after_account_merge           52
rows_after_customer_merge          52
rows_after_merchant_merge          52
unmatched_accounts                  0
transaction_ids_remain_unique    True
Name: value, dtype: object